In [10]:
from sqlalchemy import create_engine, text

# Thông tin MySQL
db_config = {
    "host": "localhost",
    "user": "myuser",
    "password": "mypassword",
    "database": "mydatabase",
}

# Kết nối MySQL
engine = create_engine(f"mysql+pymysql://{db_config['user']}:{db_config['password']}@{db_config['host']}/{db_config['database']}")


# Tạo bảng nếu chưa tồn tại
create_table_query = """
CREATE TABLE IF NOT EXISTS products (
    id INT PRIMARY KEY,
    name VARCHAR(255),
    url_key VARCHAR(255),
    url_product TEXT,
    image_base_url TEXT,
    rating_average FLOAT,
    review_count INT,
    quantity_sold VARCHAR(255),
    brand_name VARCHAR(255),
    category TEXT,
    category_id TEXT,
    current_seller VARCHAR(255),
    inventory_status VARCHAR(50),
    stock_item_qty INT,
    original_price FLOAT,
    discount FLOAT,
    price FLOAT,
    seller_rating FLOAT,
    review_count_seller INT,
    url_seller TEXT,
    seller_type VARCHAR(255)
);
"""
with engine.connect() as conn:
    conn.execute(text(create_table_query))
    conn.commit()

In [4]:
import requests
import pandas as pd
import time
import random
from tqdm import tqdm

# Phần 1: Crawl danh sách product ID
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/133.0.0.0 Safari/537.36',
}

params = {
    'limit': '10',
    'sort': 'top_seller',
    'category': '1520',
    'page': '1',
}

product_list = []
for i in range(1, 6):
    params['page'] = str(i)
    response = requests.get('https://tiki.vn/api/personalish/v1/blocks/listings', headers=headers, params=params)
    if response.status_code == 200:
        for record in response.json().get('data', []):
            product_list.append({'id': record['id']})
    time.sleep(random.uniform(3, 10))

# Lưu product ID vào MySQL
df_product_ids = pd.DataFrame(product_list)
df_product_ids.to_sql('product_ids', con=engine, if_exists='append', index=False)
print("Product IDs saved to MySQL.")

Product IDs saved to MySQL.


In [7]:
import requests
import time
import random
import pandas as pd
from tqdm import tqdm
from sqlalchemy import create_engine, text
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Kết nối MySQL
engine = create_engine(f"mysql+pymysql://{db_config['user']}:{db_config['password']}@{db_config['host']}/{db_config['database']}")
headers = {"User-Agent": "Mozilla/5.0"}

# Cấu hình session với retry
session = requests.Session()
retry_strategy = Retry(
    total=3,  # Tổng số lần thử lại
    status_forcelist=[429, 500, 502, 503, 504],  # Danh sách mã lỗi cần retry
    allowed_methods=["GET"],
    backoff_factor=1  # Khoảng thời gian delay tăng dần
)
adapter = HTTPAdapter(max_retries=retry_strategy)
session.mount("https://", adapter)

# Hàm lấy thông tin seller
def get_seller_info(seller_id):
    url = f'https://api.tiki.vn/product-detail/v2/widgets/seller?seller_id={seller_id}'
    try:
        response = session.get(url, headers=headers, timeout=5)
        if response.status_code == 200:
            seller_data = response.json().get('data', {}).get('seller', {})
            return {
                'seller_rating': seller_data.get('avg_rating_point'),
                'review_count': seller_data.get('review_count'),
                'url_seller': seller_data.get('url')
            }
    except requests.RequestException as e:
        print(f"Error fetching seller info {seller_id}: {e}")
    return {'seller_rating': None, 'review_count': None, 'url_seller': None}

# Lấy danh sách product ID từ MySQL
with engine.connect() as conn:
    result = conn.execute(text("SELECT id FROM product_ids"))
    product_ids = [row[0] for row in result]

detail_result = []
for pid in tqdm(product_ids):
    retries = 3
    while retries > 0:
        try:
            response = session.get(f'https://tiki.vn/api/v2/products/{pid}', headers=headers, timeout=5)
            if response.status_code == 200:
                data = response.json()
                seller_info = get_seller_info(data.get('current_seller', {}).get('id')) if data.get('current_seller') else {}
                detail_result.append({
                    'id': data.get('id'),
                    'name': data.get('name'),
                    'url_key': data.get('url_key'),
                    'url_product': data.get('breadcrumbs', [{}])[-1].get('url'),
                    'image_base_url': data.get('images', [{}])[0].get('base_url'),
                    'rating_average': data.get('rating_average'),
                    'review_count': data.get('review_count'),
                    'quantity_sold': data.get('quantity_sold'),
                    'brand_name': data.get('brand', {}).get('name'),
                    'category': ', '.join([bc.get('name') for bc in data.get('breadcrumbs', [])[:-1]]),
                    'category_id': ', '.join([str(bc.get('category_id')) for bc in data.get('breadcrumbs', [])[:-1]]),
                    'current_seller': data.get('current_seller', {}).get('name'),
                    'inventory_status': data.get('inventory_status'),
                    'stock_item_qty': data.get('stock_item', {}).get('qty'),
                    'original_price': data.get('original_price'),
                    'discount': data.get('discount'),
                    'price': data.get('price'),
                    'seller_rating': seller_info.get('seller_rating'),
                    'review_count_seller': seller_info.get('review_count'),
                    'url_seller': seller_info.get('url_seller'),
                    'seller_type': data.get('current_seller', {}).get('name')
                })
                break  # Nếu thành công thì thoát vòng lặp retry
            elif response.status_code == 429:
                time.sleep(random.uniform(5, 10))
        except requests.RequestException as e:
            print(f"Error fetching product {pid}: {e}")
        retries -= 1
    time.sleep(random.uniform(1, 5))

# Đổ dữ liệu vào MySQL
df_details = pd.DataFrame(detail_result)
df_details.to_sql('products', con=engine, if_exists='append', index=False)
print("Done! Product details saved to MySQL.")


100%|██████████| 50/50 [03:12<00:00,  3.84s/it]


TypeError: dict can not be used as parameter

In [12]:
import pandas as pd
import numpy as np
from sqlalchemy.types import Integer, String, Float, Boolean, DateTime

# Chuyển đổi kiểu dữ liệu
df_details = df_details.copy()

# ID sản phẩm (int)
df_details['id'] = df_details['id'].astype(int)

# Cột chuỗi (VARCHAR)
string_cols = ['name', 'url_key', 'url_product', 'image_base_url', 'brand_name', 
               'category', 'current_seller', 'inventory_status', 'url_seller', 'seller_type']
for col in string_cols:
    df_details[col] = df_details[col].astype(str).str.strip()

# Cột float
df_details['rating_average'] = df_details['rating_average'].astype(float)
df_details['seller_rating'] = df_details['seller_rating'].astype(float)

# Cột int
df_details['review_count'] = df_details['review_count'].astype(int)
df_details['review_count_seller'] = df_details['review_count_seller'].astype(int)
df_details['stock_item_qty'] = df_details['stock_item_qty'].astype(int)
df_details['original_price'] = df_details['original_price'].astype(int)
df_details['discount'] = df_details['discount'].astype(int)
df_details['price'] = df_details['price'].astype(int)

# Xử lý quantity_sold (có thể chứa dữ liệu None hoặc không phải số)
df_details['quantity_sold'] = pd.to_numeric(df_details['quantity_sold'], errors='coerce').fillna(0).astype(int)

# Xử lý category_id (nếu là số thì convert, nếu chứa ký tự thì để string)
df_details['category_id'] = pd.to_numeric(df_details['category_id'], errors='coerce')

print(df_details.dtypes)  # Kiểm tra lại kiểu dữ liệu


id                       int32
name                    object
url_key                 object
url_product             object
image_base_url          object
rating_average         float64
review_count             int32
quantity_sold            int32
brand_name              object
category                object
category_id            float64
current_seller          object
inventory_status        object
stock_item_qty           int32
original_price           int32
discount                 int32
price                    int32
seller_rating          float64
review_count_seller      int32
url_seller              object
seller_type             object
dtype: object


In [13]:
df_details.to_sql('products', con=engine, if_exists='append', index=False, dtype={
    'id': Integer,
    'name': String(255),
    'url_key': String(255),
    'url_product': String(500),
    'image_base_url': String(500),
    'rating_average': Float,
    'review_count': Integer,
    'quantity_sold': Integer,
    'brand_name': String(100),
    'category': String(255),
    'category_id': Integer,
    'current_seller': String(255),
    'inventory_status': String(50),
    'stock_item_qty': Integer,
    'original_price': Integer,
    'discount': Integer,
    'price': Integer,
    'seller_rating': Float,
    'review_count_seller': Integer,
    'url_seller': String(500),
    'seller_type': String(100),
})
print("Done! Product details saved to MySQL.")


Done! Product details saved to MySQL.
